In [ ]:
print('hello money')

In [ ]:
# ------------- configure dataframe printing layout 
import pandas as pd
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_columns', None)
#------------------------------------------------

# This ONLY affects output presentation, NOT the underlying data type
# TO FORMAT COLUMNS to display data not in scientific format
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')



In [ ]:
# FILE PATH and SQL QUERY

import numpy as np
import duckdb
from pathlib import Path
import plotly.graph_objects as go
import plotly.express as px
from binance.client import Client
from datetime import datetime
import time

# file path

folder = Path.cwd() / '_inputs'/'_1_sol_monthly_aggts_2026'

# access the files
sorted_pqt_files = sorted(folder.glob('*.parquet'))
# file list
pqt_file = [pqt.as_posix() for pqt in sorted_pqt_files]

SYMBOL = "SOLUSDT"

print(SYMBOL)
print(folder.exists())

In [ ]:
# coin stats like average volume from BINANCE

client = Client()
klines = client.futures_klines(symbol=SYMBOL, interval='1d', limit=21)
closed_klines = klines[:-1]  # Exclude today's incomplete candle
# Index 4: Close Price, Index 5: Volume
volumes = [float(k[5]) for k in closed_klines]
prices = [float(k[4]) for k in closed_klines]
# ---------------------------------
vol_20_ma = sum(volumes[-20:]) / 20
price_20_ma = sum(prices[-20:]) / 20
notional_ma = vol_20_ma * price_20_ma

vol = (f"\n volume ma for {SYMBOL} : $ {notional_ma:,.2f} \n")

print(vol)

In [ ]:
# AGGTRADES RAW TABLE

# LOAD  aggregated DATA TO dataframe with  DUCKDB

conn = duckdb.connect(database=":memory:")

# Select your desired timeframe here: '1 Day', '1 Hour', '5 Minutes', '4 Hours', etc.
timeframe = '1 Hour'

qry = f"""--sql
SELECT
    time_bucket(INTERVAL '{timeframe}', epoch_ms(CAST(transact_time AS BIGINT))) AS open_time,
    arg_min(price, transact_time) AS open,
    max(price) AS high,
    min(price) AS low,
    arg_max(price, transact_time) AS close,
    time_bucket(INTERVAL '{timeframe}', epoch_ms(CAST(transact_time AS BIGINT))) + INTERVAL '{timeframe}' - INTERVAL '1 millisecond' AS close_time,
    sum(CASE WHEN is_buyer_maker = FALSE THEN (price * quantity) ELSE 0.0 END) AS buyVol_usdt,
    sum(CASE WHEN is_buyer_maker = TRUE  THEN (price * quantity) ELSE 0.0 END) AS sellVol_usdt
FROM read_parquet({pqt_file})
GROUP BY 1
ORDER BY open_time ASC;
"""

df = conn.execute(qry).df()
bdf =df
conn.close()

# print(bdf.tail())

In [ ]:
# RAW market MEGA METRICS TABLE

bdf =df
# derived columns  ------------------------------------- PERCENTAGES
bdf['delta'] = (bdf['buyVol_usdt'] - bdf['sellVol_usdt']).round(3)

bdf['prc_change_window'] = ((
    bdf['close']-bdf['open'])/bdf['open'])*100

# Range as a percentage of the Open price
bdf['bar_volatility'] = ((bdf['high'] - bdf['low']) / bdf['open']) * 100

# day_open_price = bdf['open'].iloc[0]
# bdf['dayOpen_prcchange'] = ((
#     bdf['close'] - day_open_price)/day_open_price)*100

# core derived columns
bdf['s_delta'] = (bdf['delta']/notional_ma)*100
bdf['cumulative_sd'] = bdf['s_delta'].cumsum()

bdf['mkt_eff'] = (np.where(bdf['s_delta'] != 0,
                    (bdf['prc_change_window']/bdf['s_delta']), 0.00))
bdf['mkt_eff'] = (bdf['mkt_eff']*100).abs()

bdf['mkt_regime'] = (np.where(bdf['cumulative_sd'] != 0,
                        (bdf['prc_change_window']/bdf['cumulative_sd']), 0.00))
bdf['mkt_regime'] = (bdf['mkt_regime']*100).abs()

# ----------COLUMN FORMATING ------------------------
bdf['delta'] = bdf['delta'] #.round(2)
bdf['buyVol_usdt'] = bdf['buyVol_usdt']#.apply(lambda x: f"{x:,.2f}")
bdf['sellVol_usdt'] = bdf['sellVol_usdt']#.round(2)

# bdf.to_parquet(file_output_path)




# print(bdf.head(5))

In [ ]:
# SWING PIVOT POINTS TABLE

# ADDING THE PIVOT POINTS TO THE TABLE =====================================================

left_len =10
right_len = 10
window_size = left_len + right_len + 1
#Rolling max/min centered on the target candle
rolling_max = df['high'].rolling(window= window_size, center=True).max()
rolling_min = df['low'].rolling(window= window_size, center= True).min()
# Indentify pivot points (hing must equal max of the window and low must equal min of the window )
df['pivot_high'] = df['high'].where(df['high']==rolling_max, None)
df['pivot_low'] = df['low'].where(df['low']==rolling_min, None)
# Combined column showing the price level at the swing point, or None otherwise
df['swing_point'] = df['pivot_high'].combine_first(df['pivot_low'])

# print(df.head())

# DF for PIVOT POINTS ONLY ================================================================

pv_df = df[df['swing_point'].notna()]

# print(pv_df.to_string())

# SWING PIVOT POINTS TABLE

# 1. filter out only the candles that formed valid pivots
pivots = df[df['swing_point'].notna()].copy()
# 2. determine pivot type (high or low ) for each pivot point
pivots['type']= np.where(pivots['pivot_high'].notna(), 'high', 'low')
# 3. Reset index to calculate exact bar index distance between pivots
pivots = pivots.reset_index().rename(columns={'index':'orig_bar_index'})

# BUILD THE SWING POINTS TABLE (spt)  ========================================================
spt = pd.DataFrame()

# Start & End attributes per move (shift by -1 to pair consecutive pivots)
spt['swing_id'] = range(1, len(pivots))
spt['swing_start_time'] = pivots['open_time'][:-1].values
spt['swing_start_price']   = pivots['swing_point'][:-1].values
# spt['swing_end_time'] = pivots['open_time'][1:].values
spt['swing_end_time'] = pivots['close_time'][1:].values   
spt['swing_end_price']   = pivots['swing_point'][1:].values

#bars elapsed between start and end of the swing move
spt['bars'] = (pivots['orig_bar_index'][1:].values - pivots['orig_bar_index'][:-1].values)
#percentage price change
spt['price_change_pct'] =  ((spt['swing_end_price']- spt['swing_start_price'])/spt['swing_start_price'])*100

spt['abs_price_pct']=spt['price_change_pct'].abs()

#duration of the swing move
spt['duration']= (spt['swing_end_time']- spt['swing_start_time'])
#swing direction ; up or down
spt['direction']= np.where(spt['swing_end_price']>spt['swing_start_price'], 'up', 'down')

#Trend classification (bullish, bearish , ranging)
conditions =[spt['price_change_pct']>1.0, # bullish when change greater than 1%
             spt['price_change_pct']<-1.0 # bearish   
            ]
choices =['bullish', 'bear']

spt['trend']= np.select(conditions, choices, default='range')


# print(pivots.head())
# print(spt[150:].to_string())

# print(spt.head())


In [ ]:
# Extract bar start and end indices for every swing move from the pivots dataframe
start_indices = pivots['orig_bar_index'][:-1].values
end_indices = pivots['orig_bar_index'][1:].values

# Extract volume arrays from original dataframe (df)
df_buy_vol = df['buyVol_usdt'].values
df_sell_vol = df['sellVol_usdt'].values

# Use np.add.reduceat to compute cumulative sum between pivot indices (inclusive of start, up to end)
# Note: we sum up to end_indices + 1 if you want to include the endpoint candle in the swing volume
spt['swing_buyVol_usdt'] = [df_buy_vol[s : e + 1].sum() for s, e in zip(start_indices, end_indices)]

spt['swing_sellVol_usdt'] = [df_sell_vol[s : e + 1].sum() for s, e in zip(start_indices, end_indices)]

# Total volume
spt['swing_total_vol'] = spt['swing_buyVol_usdt'] + spt['swing_sellVol_usdt']

# Calculate Cumulative Volume Delta (CVD) for the swing move as a bonus metric
spt['swing_delta_usdt'] = spt['swing_buyVol_usdt'] - spt['swing_sellVol_usdt']

spt['swing_s_delta'] = (spt['swing_delta_usdt']/notional_ma)*100

# ----------COLUMN FORMATTING ------------------------
# Convert scientific notation to readable comma-separated strings
# spt['swing_buyVol_usdt'] = spt['swing_buyVol_usdt'].apply(lambda x: f"{x:,.2f}")
# spt['swing_sellVol_usdt'] = spt['swing_sellVol_usdt'].apply(lambda x: f"{x:,.2f}")
# spt['swing_delta_usdt'] = spt['swing_delta_usdt'].apply(lambda x: f"{x:,.2f}")

print(spt.tail(5))

In [ ]:
#  A) -- DIRECTION statistics
# LAYER   1   ***  COLUMN ANALYSIS  -- spt['price_change_pct']

x =spt['price_change_pct']

#  original percentage change data
mean =          x.mean()
median =  x.median()
max_pct =        x.max()   # Highest price change percentage
min_pct =      x.min()  # Lowest price change percentage
standard_deviation = x.std()
variance = x.var()
percentile_25th =x.quantile(0.25)
percentile_50th =x.quantile(0.5)
percentile_75th =x.quantile(0.75)
percentile_90th =x.quantile(0.90)
percentile_95th =x.quantile(0.95)

# DISPLAY
print(f" === original percentage change data -- SYMBOL :{SYMBOL} -- Timeframe :{timeframe} ======")
print(f"Average Swing: {mean:.2f} %")
print(f"Median Swing:  {median:.2f} %")
print(f"Standard dev:  {standard_deviation:.2f} %")
# print(f"Variance    :  {variance:.2f} sq %²")
print(f"\nLowest Swing:  {min_pct:.2f} %")
print(f"Highest Swing:  {max_pct:.2f} %")
print(f"\n25th percentile:  {percentile_25th:.2f} %")
print(f"50th percentile:  {percentile_50th:.2f} %")
print(f"75th percentile:  {percentile_75th:.2f} %")
print(f"90th percentile:  {percentile_90th:.2f} %")
print(f"95th percentile:  {percentile_95th:.2f} %")

In [ ]:
#  LAYER 2   ***  GROUPED DATA table (gdt) and the BELL CURVE (freq against class boundaries), modal class

# 1. Target the complete, unfiltered price change column
x = spt['price_change_pct']

# Named 'pct_range' variable to avoid overwriting Python's built-in range()
pct_range = abs(max_pct) +abs(min_pct)  
number_of_classes = 10
class_width = pct_range / number_of_classes

# 1. Define class bins using built-in range() safely
bins = [min_pct + i * class_width for i in range(number_of_classes + 1)]

# 2. Assign swings to class intervals
classes = pd.cut(x, bins=bins, include_lowest=True, right=True)

# 3. Build the Grouped Data Table (gdt)
gdt_data = []
for interval, group in spt.groupby(classes, observed=False):
    lower_limit = float(interval.left)
    upper_limit = float(interval.right)
    midpoint = (lower_limit + upper_limit) / 2.0
    frequency = len(group)
    relative_freq_pct = (frequency / len(spt)) * 100 if len(spt) > 0 else 0.0

    gdt_data.append({
        'class_interval': f"{lower_limit:.2f}% to {upper_limit:.2f}%",
        'lower_bound': lower_limit,
        'upper_bound': upper_limit,
        'midpoint': midpoint,
        'frequency': frequency,
        'relative_freq_pct': relative_freq_pct
    })

# 4. Create DataFrame
gdt = pd.DataFrame(gdt_data)

# 5. Identify Modal Class
modal_row = gdt.loc[gdt['frequency'].idxmax()]
modal_class_str = modal_row['class_interval']
modal_freq = modal_row['frequency']

print(f"=== GROUPED DATA TABLE -- SYMBOL: {SYMBOL} | Timeframe: {timeframe} ===\n")
print(gdt.to_string())
print(f"\npertange range : {pct_range} %")
print(f"Class width : {class_width.round(1)} ")
print(f"Modal Class: {modal_class_str} (Frequency: {modal_freq})")


In [ ]:
# ----- Plot Bell Curve / Distribution Chart using Plotly
fig = go.Figure()

# Bar Chart (Frequency Distribution)
fig.add_trace(go.Bar(
    x=gdt['midpoint'],
    y=gdt['frequency'],
    name='Frequency',
    text=gdt['frequency'],
    textposition='auto',
    marker_color='royalblue',
    opacity=0.75
))

# Smooth Bell Curve Line connecting midpoints
fig.add_trace(go.Scatter(
    x=gdt['midpoint'],
    y=gdt['frequency'],
    mode='lines+markers',
    name='Frequency Curve',
    line=dict(color='firebrick', width=3, shape='spline')
))

# Highlight zero point (Up vs Down boundary)
fig.add_vline(x=0, line_dash="dash", line_color="gray", annotation_text="0% Change")

fig.update_layout(
    title=f"Price Change % Distribution (Bell Curve) - {SYMBOL} ({timeframe})",
    xaxis_title="Price Change Class Midpoint (%)",
    yaxis_title="Frequency (Number of Swings)",
    template="plotly_dark",
    bargap=0.05
)

In [ ]:
#  B) -- MAGNITUDE  statistics
# LAYER   1   ***  accurate metrics

y =spt['abs_price_pct']

# ABSOLUTE   percentage change data

mean_2 =          y.mean()
median_2 =  y.median()
max_pct_2 =        y.max()   # Highest price change percentage
min_pct_2 =      y.min()   # Lowest price change percentage
standard_deviation_2 = y.std()
variance_2 = y.var()
percentile_25th_2 =y.quantile(0.25)
percentile_50th_2 =y.quantile(0.5)
percentile_75th_2 =y.quantile(0.75)
percentile_90th_2 =y.quantile(0.90)
percentile_95th_2 =y.quantile(0.95)
percentile_99th_2 =y.quantile(0.99)

# DISPLAY
print(f" === ABSOLUTE percentage change data -- SYMBOL :{SYMBOL} -- Timeframe :{timeframe} ======")
print(f"Average Swing: {mean_2:.2f} %")
print(f"Median Swing:  {median_2:.2f} %")
print(f"Standard dev:  {standard_deviation_2:.2f} %")
# print(f"Variance    :  {variance_2:.2f} sq %²")
print(f"\nLowest Swing:  {min_pct_2:.2f} %")
print(f"Highest Swing:  {max_pct_2:.2f} %")
print(f"\n25th percentile:  {percentile_25th_2:.2f} %")
print(f"50th percentile:  {percentile_50th_2:.2f} %")
print(f"75th percentile:  {percentile_75th_2:.2f} %")
print(f"90th percentile:  {percentile_90th_2:.2f} %")
print(f"95th percentile:  {percentile_95th_2:.2f} %")
print(f"99th percentile:  {percentile_99th_2:.2f} %")

In [ ]:
#  LAYER 2   ***  GROUPED DATA table (gdt_2)

# 1. Target the absolute change column
y =spt['abs_price_pct']

# Named 'pct_range' variable to avoid overwriting Python's built-in range()
pct_range_2 = max_pct_2 + min_pct_2
number_of_classes_2 = 10
class_width_2 = pct_range_2 / number_of_classes_2

# 1. Define class bins using built-in range() safely
bins_2 = [min_pct_2 + int * class_width_2 for int in range(number_of_classes_2 + 1)]

# 2. Assign swings to class intervals
classes = pd.cut(y, bins=bins_2, include_lowest=True, right=True)

# 3. Build the Grouped Data Table (gdt)
gdt_2_data = []
for interval, group in spt.groupby(classes, observed=False):
    lower_limit = float(interval.left)
    upper_limit = float(interval.right)
    midpoint = (lower_limit + upper_limit) / 2.0
    frequency = len(group)
    relative_freq_pct = (frequency / len(spt)) * 100 if len(spt) > 0 else 0.0

    gdt_2_data.append({
        'class_interval': f"{lower_limit:.2f}% to {upper_limit:.2f}%",
        'lower_bound': lower_limit,
        'upper_bound': upper_limit,
        'midpoint': midpoint,
        'frequency': frequency,
        'relative_freq_pct': relative_freq_pct
    })

# 4. Create DataFrame
gdt_2 = pd.DataFrame(gdt_2_data)

# 5. Identify Modal Class
modal_row = gdt_2.loc[gdt_2['frequency'].idxmax()]
modal_class_str = modal_row['class_interval']
modal_freq = modal_row['frequency']

print(f"=== GROUPED DATA TABLE -- SYMBOL: {SYMBOL} | Timeframe: {timeframe} ===\n")
print(gdt_2.to_string())
print(f"\nrange : {pct_range_2:.2f} %")
print(f"Class width : {class_width_2.round(1)} ")
print(f"Modal Class: {modal_class_str} (Frequency: {modal_freq})")

In [ ]:
# ----- Plot Bell Curve / Distribution Chart using Plotly
fig_2 = go.Figure()

# Bar Chart (Frequency Distribution)
fig_2.add_trace(go.Bar(
    x=gdt_2['midpoint'],
    y=gdt_2['frequency'],
    name='Frequency',
    text=gdt_2['frequency'],
    textposition='auto',
    marker_color='royalblue',
    opacity=0.75
))

# Smooth Bell Curve Line connecting midpoints
fig_2.add_trace(go.Scatter(
    x=gdt_2['midpoint'],
    y=gdt_2['frequency'],
    mode='lines+markers',
    name='Frequency Curve',
    line=dict(color='firebrick', width=3, shape='spline')
))

# Highlight zero point (Up vs Down boundary)
fig_2.add_vline(x=0, line_dash="dash", line_color="gray", annotation_text="0% Change")

fig_2.update_layout(
    title=f"Price Change % Distribution (Bell Curve) - {SYMBOL} ({timeframe})",
    xaxis_title="Price Change Class Midpoint (%)",
    yaxis_title="Frequency (Number of Swings)",
    template="plotly_dark",
    bargap=0.05
)

In [ ]:
# C) -- DURATION  statistics
# LAYER   1   ***  COLUMN ANALYSIS  -- spt['bars']

b = spt['bars']

mean_3 =          b.mean()
median_3 =  b.median()
max_pct_3 =        b.max()  
min_pct_3 =      b.min()   
standard_deviation_3 = b.std()
variance_3 = b.var()
percentile_25th_3 =b.quantile(0.25)
percentile_50th_3 =b.quantile(0.5)
percentile_75th_3 =b.quantile(0.75)
percentile_90th_3 =b.quantile(0.90)
percentile_99th_3 =b.quantile(0.99)

# DISPLAY
print(f" ============  Number of {timeframe} bars ======  SYMBOL: {SYMBOL}")
print(f"\nAverage duration: {mean_3:.2f} bars")
print(f"Median duration:  {median_3:.2f} bars")
print(f"Standard dev duration: {standard_deviation_3:.2f} bars")
# print(f"Duration Variance    : {variance:.2f} bars")
print(f"\nLowest duration   :  {min_pct_3:.2f} bars")
print(f"Highest duration    :  {max_pct_3:.2f} bars")
print(f"\n25th percentile:  {percentile_25th_3:.2f} bars")
print(f"50th percentile:  {percentile_50th_3:.2f} bars")
print(f"75th percentile:  {percentile_75th_3:.2f} bars")
print(f"90th percentile:  {percentile_90th_3:.2f} bars")
print(f"99th percentile:  {percentile_99th_3:.2f} bars")


In [ ]:
#  LAYER 2   ***  GROUPED DATA table (gdt_2)

# 1. Target the absolute change column
b = spt['bars']

# Named 'pct_range' variable to avoid overwriting Python's built-in range()
pct_range_3 = max_pct_3 + min_pct_3
number_of_classes_3 = 10
class_width_3 = pct_range_3 / number_of_classes_3

# 1. Define class bins using built-in range() safely
bins_3 = [min_pct_3 + num * class_width_3 for num in range(number_of_classes_3 + 1)]

# 2. Assign swings to class intervals
classes = pd.cut(b, bins=bins_3, include_lowest=True, right=True)

# 3. Build the Grouped Data Table (gdt)
gdt_3_data = []
for interval, group in spt.groupby(classes, observed=False):
    lower_limit = float(interval.left)
    upper_limit = float(interval.right)
    midpoint = (lower_limit + upper_limit) / 2.0
    frequency = len(group)
    relative_freq_pct = (frequency / len(spt)) * 100 if len(spt) > 0 else 0.0

    gdt_3_data.append({
        'class_interval': f"{lower_limit:.2f} to {upper_limit:.2f} bars",
        'lower_bound': lower_limit,
        'upper_bound': upper_limit,
        'midpoint': midpoint,
        'frequency': frequency,
        'relative_freq_pct': relative_freq_pct
    })

# 4. Create DataFrame
gdt_3 = pd.DataFrame(gdt_3_data)

# 5. Identify Modal Class
modal_row = gdt_3.loc[gdt_3['frequency'].idxmax()]
modal_class_str = modal_row['class_interval']
modal_freq = modal_row['frequency']

print(f"=== GROUPED DATA TABLE -- SYMBOL: {SYMBOL} | Timeframe: {timeframe} ===\n")
print(gdt_3.to_string())
print(f"\nrange : {pct_range_3:.2f} bars")
print(f"Class width : {class_width_3.round(1)} ")
print(f"Modal Class: {modal_class_str} (Frequency: {modal_freq})")

In [ ]:
# ----- Plot Bell Curve / Distribution Chart using Plotly
fig_3 = go.Figure()

# Bar Chart (Frequency Distribution)
fig_3.add_trace(go.Bar(
    x=gdt_3['midpoint'],
    y=gdt_3['frequency'],
    name='Frequency',
    text=gdt_3['frequency'],
    textposition='auto',
    marker_color='royalblue',
    opacity=0.75
))

# Smooth Bell Curve Line connecting midpoints
fig_3.add_trace(go.Scatter(
    x=gdt_3['midpoint'],
    y=gdt_3['frequency'],
    mode='lines+markers',
    name='Frequency Curve',
    line=dict(color='firebrick', width=3, shape='spline')
))

# Highlight zero point (Up vs Down boundary)
# fig_3.add_vline(x=0, line_dash="dash", line_color="gray", annotation_text="0 bars")

fig_3.update_layout(
    title=f"Distribution of trade swings' duration - {SYMBOL} ",
    xaxis_title=f"Duration in ({timeframe}) bars ",
    yaxis_title="Frequency (Number of Swings)",
    template="plotly_dark",
    bargap=0.05
)